# 🤖 Yapay Zeka ve Üretken Modeller Atölyesi - 3
**BTK Akademi | Dr. Öğr. Üyesi Furkan Göz**

---

## 📚 Bu Haftanın Konuları

1. Derin Öğrenmenin Yükselişi & AlexNet/ImageNet
2. Yapay Sinir Ağları (ANN) — Temel Kavramlar
3. Aktivasyon Fonksiyonları
4. Veri Ön İşleme (Data Preprocessing)
5. Keras ile Model Tanımı, Derleme ve Eğitim
6. Kayıp Fonksiyonları & Optimizasyon
7. Regularization (Dropout & L2)
8. Backpropagation & Gradient Descent
9. Hiperparametreler (Epoch, Batch Size, Learning Rate)
10. Eğitim / Doğrulama / Test Ayrımı & Early Stopping
11. **Proje 1:** Iris Çiçeği Sınıflandırma (MLP)
12. **Proje 2:** Fashion MNIST Görüntü Sınıflandırma (MLP)

## 0. Gerekli Kütüphanelerin Kurulumu ve İçe Aktarımı

In [ ]:
# Gerekli kütüphaneleri kur (Colab veya yerel ortam için)
# !pip install tensorflow scikit-learn matplotlib numpy pandas

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, InputLayer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import fashion_mnist

print(f'TensorFlow sürümü: {tf.__version__}')
print('Tüm kütüphaneler başarıyla yüklendi ✅')

---
## 1. Derin Öğrenme & AlexNet / ImageNet

### 📌 Temel Bilgiler
- **ImageNet**: 14 milyon, 1000 sınıflı görüntü içeren benchmark veri setidir.
- **AlexNet (2012)**: Alex Krizhevsky, Ilya Sutskever ve Geoffrey Hinton tarafından geliştirildi. GPU kullanan ilk büyük ölçekli derin öğrenme modeli. ImageNet hata oranını **~%26 → ~%15** düşürdü.
- Bu başarı + artan GPU gücü + Big Data erişilebilirliği → **Derin Öğrenme çağı**

### AlexNet Mimarisi (Özet)
```
Input (227x227x3)
  → CONV(11x11, stride=4, 96 kernel) → MaxPool
  → CONV(5x5, pad=2, 256 kernel)     → MaxPool  
  → CONV(3x3, pad=1, 384 kernel)
  → CONV(3x3, pad=1, 384 kernel)
  → CONV(3x3, pad=1, 256 kernel)     → MaxPool
  → FC(4096) → FC(4096) → FC(1000, Softmax)
```

---
## 2. Yapay Sinir Ağı (ANN) — Temel Kavramlar

### Yapay Nöron Matematiği

$$z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b \quad \Rightarrow \quad a = f(z)$$

- $x_i$: Girdiler (Features)
- $w_i$: Ağırlıklar (öğrenilen önem dereceleri)
- $b$: Bias (sapma) — karar sınırının esnekliğini sağlar
- $f(z)$: Aktivasyon fonksiyonu

### Katman Türleri
| Katman | Görevi |
|--------|--------|
| **Giriş (Input)** | Ham veriyi ağa iletir; hesaplama yapılmaz |
| **Gizli (Hidden)** | Özellik çıkarımı (feature extraction) yapar |
| **Çıkış (Output)** | Sınıflandırma veya regresyon tahmini üretir |

### Boyutlandırma Kuralları
- **Giriş nöron sayısı** = Feature (sütun) sayısı
- **Çıkış nöron sayısı** = Regresyon → 1 | Binary → 1 (Sigmoid) | Multiclass → K sınıf (Softmax)

In [ ]:
# --- Örnek: Iris veri setinde giriş/çıkış boyutlarını otomatik belirleme ---

data = load_iris()
X = data.data
y = data.target

input_dim  = X.shape[1]          # Sütun sayısı = özellik sayısı
output_dim = len(np.unique(y))   # Benzersiz sınıf sayısı

print(f'Girdi Nöronu Sayısı : {input_dim}')   # → 4
print(f'Çıkış Nöronu Sayısı : {output_dim}')  # → 3
print(f'Özellik isimleri    : {data.feature_names}')
print(f'Sınıf isimleri      : {data.target_names}')

In [ ]:
# --- İleri Yayılım (Forward Propagation) Sayısal Örneği ---
# Girdiler ve öğrenilmiş ağırlıklar
x1, x2 = 2, 3       # Yaş, Kilo
w1, w2 = 0.5, 0.3   # Ağırlıklar
b = 2               # Bias

# Adım 1: Ağırlıklı lineer toplam
z = w1 * x1 + w2 * x2 + b
print(f'z = {w1}×{x1} + {w2}×{x2} + {b} = {z}')

# Adım 2: Sigmoid aktivasyon
import math
sigmoid = lambda z: 1 / (1 + math.exp(-z))
a = sigmoid(z)
print(f'f(z) = Sigmoid({z}) ≈ {a:.4f}')
print(f'Yorum: Model bu verinin hedef sınıfa ait olma olasılığını %{a*100:.1f} olarak tahmin ediyor.')

---
## 3. Aktivasyon Fonksiyonları

Aktivasyonlar ağa **doğrusal olmayanlık (non-linearity)** kazandırır. 
Aktivasyon **olmadan** ne kadar derin olursa olsun ağ, tek katmanlı lineer regresyona indirger:
$$f(f(x)) = W(Wx + b) + b \Rightarrow \text{tek lineer dönüşüm}$$

| Fonksiyon | Formül | Aralık | Kullanım Yeri |
|-----------|--------|--------|---------------|
| Sigmoid | $\frac{1}{1+e^{-x}}$ | (0,1) | Binary çıkış |
| Tanh | $\tanh(x)$ | (-1,1) | Gizli katman (eski) |
| **ReLU** | $\max(0,x)$ | $[0,\infty)$ | **Gizli katman (varsayılan)** |
| Leaky ReLU | $x$ if $x>0$ else $\alpha x$ | $(-\infty,\infty)$ | ReLU alternatifi |
| GELU | — | — | Transformer mimarileri |
| **Softmax** | $\frac{e^{z_i}}{\sum_j e^{z_j}}$ | (0,1), toplam=1 | **Multiclass çıkış** |

In [ ]:
# --- Aktivasyon Fonksiyonlarını Görselleştirme ---

x = np.linspace(-6, 6, 300)

sigmoid_fn    = 1 / (1 + np.exp(-x))
tanh_fn       = np.tanh(x)
relu_fn       = np.maximum(0, x)
leaky_relu_fn = np.where(x > 0, x, 0.1 * x)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Aktivasyon Fonksiyonları', fontsize=14, fontweight='bold')

funcs = [
    (sigmoid_fn,    'Sigmoid',     'royalblue',    '$(0, 1)$ — Binary çıkış'),
    (tanh_fn,       'Tanh',        'darkorange',   '$(-1, 1)$ — Eski gizli katman'),
    (relu_fn,       'ReLU',        'green',        '$[0, \\infty)$ — Varsayılan gizli'),
    (leaky_relu_fn, 'Leaky ReLU',  'crimson',      '$(-\\infty, \\infty)$ — ReLU alt.'),
]

for ax, (fn, name, color, desc) in zip(axes.flat, funcs):
    ax.plot(x, fn, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(f'{name}\n{desc}', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-6, 6)

plt.tight_layout()
plt.savefig('aktivasyon_fonksiyonlari.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Softmax Örneği ---
# Ham logit skorları (modelin ürettiği çıktılar)
logits = np.array([1.3, 5.1, 2.2, 0.7, 1.1])

def softmax(z):
    e_z = np.exp(z - np.max(z))  # sayısal kararlılık için max çıkarılır
    return e_z / e_z.sum()

probabilities = softmax(logits)

print('Logitler (ham skorlar) :', logits)
print('Softmax olasılıkları   :', np.round(probabilities, 4))
print(f'Toplam olasılık        : {probabilities.sum():.4f}  (her zaman 1.0)')
print(f'Tahmin edilen sınıf    : {np.argmax(probabilities)} ({probabilities.max()*100:.1f}% olasılıkla)')

---
## 4. Veri Ön İşleme (Data Preprocessing)

Ham veri → model için hazır veri dönüşümünün temel adımları:

1. **Eksik veri temizleme** — NaN değerlerin doldurulması / silinmesi  
2. **Etiketleme** — Kategorik değişkenler → sayısal (Label/One-Hot Encoding)  
3. **Özellik ölçekleme** — Tüm sayısal verileri ortak ölçeğe getirme  
4. **Normalizasyon / Standartlaştırma** — Varyans ve ortalama düzenleme  
5. **Veri bölme** — Train / Validation / Test ayrımı  

> ⚠️ **Kritik kural:** `StandardScaler` (ve tüm ön işleme adımları) yalnızca `X_train` üzerinde `fit` edilmeli, `X_test` üzerinde sadece `transform` uygulanmalıdır. Aksi hâlde **data leakage** (veri sızıntısı) oluşur.

In [ ]:
# --- Keras için Iris Veri Ön İşleme ---

X, y = load_iris(return_X_y=True)

# One-Hot Encoding: [0,1,2] → [[1,0,0],[0,1,0],[0,0,1]]
y_ohe = to_categorical(y)
print('Ham etiket örneği  :', y[:5])
print('OHE etiket örneği  :')
print(y_ohe[:5])

# Train-Test bölme
X_train, X_test, y_train, y_test = train_test_split(
    X, y_ohe, test_size=0.2, random_state=42, stratify=y
)

# Standartlaştırma (fit sadece train üzerinde!)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'\nX_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} | y_test: {y_test.shape}')
print(f'X_train ortalama ≈ {X_train.mean():.4f} (≈0 olmalı)')
print(f'X_train std      ≈ {X_train.std():.4f}  (≈1 olmalı)')

---
## 5. Kayıp Fonksiyonları (Loss Functions)

Kayıp fonksiyonu, model tahmini ile gerçek değer arasındaki farkı ölçer. Eğitimin amacı bu değeri **minimize** etmektir.

| Problem Türü | Çıkış Aktivasyonu | Kayıp Fonksiyonu |
|---|---|---|
| Regresyon | Linear | Mean Squared Error (MSE) |
| Binary Sınıflandırma | Sigmoid | Binary Crossentropy |
| Multiclass (One-Hot y) | Softmax | Categorical Crossentropy |
| Multiclass (Integer y) | Softmax | Sparse Categorical Crossentropy |

In [ ]:
# --- Gradient Descent Sayısal Örneği (1 adım) ---

x = 2        # Girdi
w = 0.3      # Başlangıç ağırlığı
y_true = 1   # Gerçek değer
lr = 0.1     # Öğrenme hızı (learning rate)

# 1. İleri yayılım
y_pred = w * x
print(f'Tahmin (ŷ)         : w × x = {w} × {x} = {y_pred}')

# 2. Kayıp (MSE)
loss = 0.5 * (y_true - y_pred) ** 2
print(f'Kayıp (L)          : 0.5 × ({y_true} - {y_pred})² = {loss}')

# 3. Gradyan
grad = (y_pred - y_true) * x
print(f'Gradyan (∂L/∂w)    : ({y_pred} - {y_true}) × {x} = {grad}')

# 4. Ağırlık güncelleme
w_new = w - lr * grad
print(f'Yeni ağırlık (w)   : {w} - {lr} × ({grad}) = {w_new}')
print(f'\nYeni tahmin        : {w_new} × {x} = {w_new * x}  (gerçeğe daha yakın!)')

---
## 6. Optimizasyon Algoritmaları

| Optimizer | Hız | Bellek | Uyarlamalı Adım | Kararlılık | Not |
|-----------|-----|--------|-----------------|------------|-----|
| **SGD** | Yavaş | Az | ❌ | Düşük | Local minima riski |
| **RMSprop** | Orta | Orta | ✅ | Orta | RNN'lerde başarılı |
| **Adam** | Hızlı | Orta | ✅ | Yüksek | **Çoğu problemde varsayılan** |

**Adam** = Momentum + RMSprop → adaptif öğrenme hızı + momentumu birleştirir.

In [ ]:
# --- Learning Rate Etkisini Görselleştirme ---

x_vals = np.linspace(-3, 3, 300)
loss_curve = x_vals ** 2  # Basit kare hata yüzeyi

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Learning Rate Etkisi', fontsize=13, fontweight='bold')

scenarios = [
    ('Çok Küçük LR (0.001)', 2.5, 0.001, 'steelblue'),
    ('Uygun LR (0.1)',        2.5, 0.1,   'green'),
    ('Çok Büyük LR (0.9)',   2.5, 0.9,   'crimson'),
]

for ax, (title, start, lr_val, color) in zip(axes, scenarios):
    ax.plot(x_vals, loss_curve, 'gray', linewidth=2, label='Kayıp yüzeyi')
    
    pos = start
    steps_x = [pos]
    for _ in range(8):
        grad = 2 * pos  # Türev: d(x²)/dx = 2x
        pos = pos - lr_val * grad
        steps_x.append(pos)
    
    steps_y = [p**2 for p in steps_x]
    ax.plot(steps_x, steps_y, 'o-', color=color, linewidth=2, markersize=6)
    ax.set_title(title)
    ax.set_xlabel('Ağırlık (θ)')
    ax.set_ylabel('Kayıp J(θ)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_rate_etkisi.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Regularization (Düzenlileştirme)

Overfitting'i önlemek için kullanılan teknikler:

- **L1 (Lasso):** Kayıp fonksiyonuna $\lambda \sum |w_i|$ eklenir → seyrek ağırlıklar
- **L2 (Ridge):** Kayıp fonksiyonuna $\lambda \sum w_i^2$ eklenir → küçük ağırlıklar
- **Dropout:** Eğitim sırasında nöronların rastgele `p` olasılıkla devre dışı bırakılması

---
## 8. Hiperparametreler

| Hiperparametre | Açıklama | Tipik Değerler |
|---|---|---|
| **Epoch** | Tüm veri setinin modelden kaç kez geçirileceği | 10 – 200 |
| **Batch Size** | Ağırlık güncellemesi öncesi işlenen örnek sayısı | 8, 16, 32, 64, 128 |
| **Learning Rate** | Her güncelleme adımının büyüklüğü | 0.001 (Adam varsayılan) |
| **Dropout Rate** | Eğitimde kapatılan nöron oranı | 0.2 – 0.5 |
| **Hidden Layer** | Gizli katman sayısı ve nöron sayısı | Deneme-yanılma / AutoML |

---
## 🔬 Proje 1: Iris Çiçeği Sınıflandırma (MLP)

**Görev:** 4 özellik kullanarak 3 iris türünü sınıflandır (Setosa, Versicolor, Virginica)

**Mimari:** Input(4) → Dense(64, ReLU) + Dropout(0.5) + L2 → Dense(3, Softmax)

In [ ]:
# ================================================
#  PROJE 1: IRIS ÇİÇEĞİ SINIFLANDIRMA
# ================================================

# --- Adım 1: Veri Yükleme ve Ön İşleme ---
X, y = load_iris(return_X_y=True)
y_ohe = to_categorical(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_ohe, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Veri hazırlama tamamlandı!')
print(f'Eğitim: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# --- Adım 2: Model Tanımı (Dropout + L2 Regularization) ---
tf.random.set_seed(42)

model_iris = Sequential([
    InputLayer(input_shape=(4,)),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),  # Eğitimde %50 nöron kapatılır → overfitting azalır
    Dense(3, activation='softmax')
], name='Iris_MLP')

model_iris.summary()

In [ ]:
# --- Adım 3: Model Derleme ---
model_iris.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping: val_loss 5 epoch iyileşmezse dur, en iyi ağırlıkları geri yükle
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
# --- Adım 4: Model Eğitimi ---
history_iris = model_iris.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# --- Adım 5: Model Değerlendirme ---
test_loss, test_acc = model_iris.evaluate(X_test, y_test, verbose=0)
print(f'Test Kaybı   : {test_loss:.4f}')
print(f'Test Doğruluğu: {test_acc:.4f} ({test_acc*100:.1f}%)')

In [ ]:
# --- Adım 6: Öğrenme Eğrisi Grafikleri ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Iris MLP — Öğrenme Eğrileri', fontsize=13, fontweight='bold')

# Kayıp grafiği
axes[0].plot(history_iris.history['loss'],     label='Eğitim Kaybı',     color='steelblue')
axes[0].plot(history_iris.history['val_loss'], label='Doğrulama Kaybı',  color='darkorange')
axes[0].set_title('Kayıp (Loss)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Kayıp Değeri')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Doğruluk grafiği
axes[1].plot(history_iris.history['accuracy'],     label='Eğitim Doğruluğu',    color='green')
axes[1].plot(history_iris.history['val_accuracy'], label='Doğrulama Doğruluğu', color='crimson')
axes[1].set_title('Doğruluk (Accuracy)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Doğruluk')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('iris_ogrenme_egrisi.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Adım 7: Yeni Örnek Üzerinde Tahmin ---
species = ['Setosa', 'Versicolor', 'Virginica']

sample = X_test[0].reshape(1, -1)
prediction = model_iris.predict(sample, verbose=0)
true_class = np.argmax(y_test[0])

print('Tahmin Olasılıkları:')
for i, (name, prob) in enumerate(zip(species, prediction[0])):
    marker = '← TAHMİN' if i == np.argmax(prediction) else ''
    print(f'  {name:12s}: %{prob*100:.1f}  {marker}')
print(f'\nGerçek Sınıf: {species[true_class]}')

---
## 🔬 Proje 2: Fashion MNIST Görüntü Sınıflandırma (MLP)

**Veri Seti:** 70.000 adet 28×28 gri tonlamalı giysi görseli, 10 sınıf  
**Görev:** Giysi türünü sınıflandır (T-shirt, Trouser, Pullover, ...)

| Label | Sınıf | Label | Sınıf |
|-------|-------|-------|-------|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

In [ ]:
# ================================================
#  PROJE 2: FASHION MNIST GÖRÜNTÜ SINIFLANDIRMA
# ================================================

class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

# --- Adım 1: Veri Yükleme ---
(X_train_fm, y_train_fm), (X_test_fm, y_test_fm) = fashion_mnist.load_data()

print(f'Eğitim seti: {X_train_fm.shape}  — {y_train_fm.shape}')
print(f'Test seti  : {X_test_fm.shape}  — {y_test_fm.shape}')

In [ ]:
# --- Veri Setinden Örnek Görseller ---
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
fig.suptitle('Fashion MNIST — Örnek Görseller', fontsize=12, fontweight='bold')

# Her sınıftan birer örnek seç
for cls_idx in range(10):
    idx = np.where(y_train_fm == cls_idx)[0][0]
    ax = axes[cls_idx // 5][cls_idx % 5]
    ax.imshow(X_train_fm[idx], cmap='gray')
    ax.set_title(f'{cls_idx}: {class_names[cls_idx]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('fashion_mnist_ornekler.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Adım 2: Veri Ön İşleme ---

# Normalizasyon: piksel değerlerini [0, 255] → [0, 1]
X_train_fm = X_train_fm / 255.0
X_test_fm  = X_test_fm  / 255.0

# Düzleştirme (Flatten): 28×28 matris → 784 boyutlu vektör
X_train_flat = X_train_fm.reshape(-1, 784)
X_test_flat  = X_test_fm.reshape(-1, 784)

# One-Hot Encoding
y_train_ohe = to_categorical(y_train_fm, 10)
y_test_ohe  = to_categorical(y_test_fm, 10)

print(f'X_train (düzleştirilmiş): {X_train_flat.shape}')
print(f'y_train (one-hot)        : {y_train_ohe.shape}')
print(f'Piksel aralığı           : [{X_train_flat.min():.1f}, {X_train_flat.max():.1f}]')

In [ ]:
# --- Adım 3: MLP Modeli Oluşturma ---
tf.random.set_seed(42)

model_fm = Sequential([
    Dense(256, activation='relu', input_shape=(784,)),
    Dropout(0.4),   # %40 nöron rastgele kapatılır
    Dense(128, activation='relu'),
    Dropout(0.3),   # %30 nöron rastgele kapatılır
    Dense(10, activation='softmax')
], name='FashionMNIST_MLP')

model_fm.summary()

In [ ]:
# --- Adım 4: Model Derleme ---
model_fm.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_fm = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
# --- Adım 5: Model Eğitimi ---
history_fm = model_fm.fit(
    X_train_flat, y_train_ohe,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop_fm],
    verbose=1
)

In [ ]:
# --- Adım 6: Test Değerlendirmesi ---
test_loss_fm, test_acc_fm = model_fm.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f'Test Kaybı   : {test_loss_fm:.4f}')
print(f'Test Doğruluğu: {test_acc_fm:.4f} ({test_acc_fm*100:.2f}%)')
# Fashion MNIST için tipik MLP başarımı: ~%87-89

In [ ]:
# --- Adım 7: Öğrenme Eğrisi Grafikleri ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Fashion MNIST MLP — Öğrenme Eğrileri', fontsize=13, fontweight='bold')

axes[0].plot(history_fm.history['loss'],     label='Eğitim Kaybı',     color='steelblue')
axes[0].plot(history_fm.history['val_loss'], label='Doğrulama Kaybı',  color='darkorange')
axes[0].set_title('Kayıp (Loss)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Kayıp Değeri')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_fm.history['accuracy'],     label='Eğitim Doğruluğu',    color='green')
axes[1].plot(history_fm.history['val_accuracy'], label='Doğrulama Doğruluğu', color='crimson')
axes[1].set_title('Doğruluk (Accuracy)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Doğruluk')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fashion_ogrenme_egrisi.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- Adım 8: Tahmin Görselleştirme ---
# Rastgele 10 test örneği için tahmin yap ve görselleştir
np.random.seed(42)
sample_indices = np.random.choice(len(X_test_flat), 10, replace=False)

predictions = model_fm.predict(X_test_flat[sample_indices], verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Fashion MNIST — Tahmin Sonuçları', fontsize=12, fontweight='bold')

for i, (ax, idx) in enumerate(zip(axes.flat, sample_indices)):
    img = X_test_fm[idx]
    true_label  = y_test_fm[idx]
    pred_label  = np.argmax(predictions[i])
    confidence  = predictions[i][pred_label]
    
    ax.imshow(img, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(
        f'Gerçek: {class_names[true_label]}\n'
        f'Tahmin: {class_names[pred_label]} (%{confidence*100:.0f})',
        fontsize=7.5, color=color
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('fashion_tahminler.png', dpi=120, bbox_inches='tight')
plt.show()
print('Yeşil başlık = Doğru tahmin | Kırmızı başlık = Yanlış tahmin')

---
## 📋 Hafta Özeti

| Konu | Özet |
|------|------|
| **ANN Temeli** | $z = Wx + b$, aktivasyon ile non-linearity |
| **Aktivasyonlar** | Gizli → ReLU, Binary çıkış → Sigmoid, Multiclass → Softmax |
| **Preprocessing** | Scaler yalnızca train üzerinde fit edilmeli |
| **Loss Seçimi** | One-Hot → `categorical_crossentropy` \| Integer → `sparse_categorical_crossentropy` |
| **Optimizer** | Adam çoğu problemde varsayılan seçim |
| **Regularization** | Dropout + L2 → overfitting azaltır |
| **Early Stopping** | `val_loss` izle, `patience` epoch bekle |
| **LR** | Çok küçük → yavaş, çok büyük → diverjans |
| **Backpropagation** | Forward pass → Loss → Backward pass → Weight update |

---

### 🔗 Kaynaklar
- [TensorFlow/Keras Dokümantasyonu](https://www.tensorflow.org/api_docs/python/tf/keras)
- [BTK Akademi](https://www.btkakademi.gov.tr)
- Krizhevsky, A. et al. (2012). *ImageNet Classification with Deep CNNs* (AlexNet)

---
*Bu notebook BTK Akademi — Yapay Zeka ve Üretken Modeller Atölyesi (Hafta 3) kapsamında hazırlanmıştır.*